In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import hashlib
import ast
import re

In [2]:
FILE = '2026-04-19.xlsx' 
path = Path()
file_path = path.absolute() / 'out' / FILE

In [3]:
EXCLUDED_WEIGHTS = [
    '[1, 0, 0, 0, 0]',
    '[0, 1, 0, 0, 0]',
    '[0, 0, 1, 0, 0]',
    '[0, 0, 0, 1, 0]',
    '[0, 0, 0, 0, 1]',
]

In [4]:
df = pd.read_excel(file_path, engine='openpyxl')

In [5]:
df = df[~df["weight"].isin(EXCLUDED_WEIGHTS)].copy()

In [6]:
# pattern = r"PRP(\d+)_C(\d+)_P(\d+)_V(\d+)_T(\d+)_S(\d+)"
# matches = df['file'].str.extract(pattern)
# matches.columns = ['instancia', 'clientes', 'produtos', 'veiculos', 'periodos', 'seeds']
# df = pd.concat([df, matches], axis=1)

In [7]:
df[df['instancia'].isna()].iloc[:, 0].count()

np.int64(336)

In [8]:
df = df[~df['instancia'].isna()]

In [9]:
hash_list = df["weight_hash"].unique()

def make_weight_hash_map_from_list(hash_list, start_at=1):
    return {h: rf"$w_{{{i}}}$" for i, h in enumerate(hash_list, start=start_at)}

hash_map = make_weight_hash_map_from_list(hash_list)

In [10]:
df["weight_label"] = df["weight_hash"].map(hash_map).fillna(df["weight_hash"])

In [11]:
hash_map

{'d086fe': '$w_{1}$',
 'edcc2b': '$w_{2}$',
 'acdb7b': '$w_{3}$',
 '6d360f': '$w_{4}$',
 'c264b9': '$w_{5}$',
 '272ddf': '$w_{6}$'}

In [12]:
df['instancia'] = df['instancia'].astype(int)

In [13]:
df['desv_rel_targ_1'] = df['p1'] / df['f1_target']
df['desv_rel_targ_2'] = df['p2'] / df['f2_target']
df['desv_rel_targ_3'] = df['p3'] / df['f3_target']
df['desv_rel_targ_4'] = df['p4'] / df['f4_target']
df['desv_rel_targ_5'] = df['p5'] / df['f5_target']

In [14]:
df_min_desv_rel = pd.pivot_table(
    df,
    index=['file', 'time'],
    aggfunc={
        'desv_rel_targ_1': 'min',
        'desv_rel_targ_2': 'min',
        'desv_rel_targ_3': 'min',
        'desv_rel_targ_4': 'min',
        'desv_rel_targ_5': 'min'},
).rename(columns={
    'desv_rel_targ_1': 'min_desv_rel_targ_1',
    'desv_rel_targ_2': 'min_desv_rel_targ_2',
    'desv_rel_targ_3': 'min_desv_rel_targ_3',
    'desv_rel_targ_4': 'min_desv_rel_targ_4',
    'desv_rel_targ_5': 'min_desv_rel_targ_5'})

In [15]:
df_min = pd.merge(
    df,
    df_min_desv_rel,
    on=['file', 'time'],
)

In [16]:
df_min['ratio_1'] = df_min['desv_rel_targ_1'] / df_min['min_desv_rel_targ_1']
df_min['ratio_2'] = df_min['desv_rel_targ_2'] / df_min['min_desv_rel_targ_2']
df_min['ratio_3'] = df_min['desv_rel_targ_3'] / df_min['min_desv_rel_targ_3']
df_min['ratio_4'] = df_min['desv_rel_targ_4'] / df_min['min_desv_rel_targ_4']
df_min['ratio_5'] = df_min['desv_rel_targ_5'] / df_min['min_desv_rel_targ_5']

In [17]:
df_min[['ratio_1', 'ratio_2', 'ratio_3', 'ratio_4', 'ratio_5']] = df_min[['ratio_1', 'ratio_2', 'ratio_3', 'ratio_4', 'ratio_5']].fillna(1)

In [18]:
df_min_clean = df_min[df_min['ratio_1'] < 4]

In [19]:
taus = np.arange(1, 4, 0.001)
ratio_cols = [f'ratio_{i}' for i in range(1, 6)]
total_experiments = df_min_clean.shape[0]

df_long = df_min_clean.melt(
    id_vars=['weight_label', 'alpha'],
    value_vars=ratio_cols,
    var_name='ratio',
    value_name='ratio_value'
)

out = (
    df_long.groupby(['weight_label', 'alpha'])['ratio_value']
    .apply(lambda s: pd.Series({tau: (s <= tau).sum() for tau in taus}))
    .rename('count')
    .reset_index()
    .rename(columns={'level_2': 'tau'})
)

# out: weight | alpha | tau | count

In [20]:
# Normalizar por ratio: dividir pelo máximo `count` dentro de cada `ratio`
out['count_norm'] = out['count'] / out['count'].max()
# Checagens (esperado: 1.0)
# print(out.loc[out['ratio'] == 'ratio_1', 'count_norm'].max())
# print(out.loc[out['ratio'] == 'ratio_2', 'count_norm'].max())
# print(out.loc[out['ratio'] == 'ratio_3', 'count_norm'].max())
# print(out.loc[out['ratio'] == 'ratio_4', 'count_norm'].max())
# print(out.loc[out['ratio'] == 'ratio_5', 'count_norm'].max())

In [21]:
hash_map

{'d086fe': '$w_{1}$',
 'edcc2b': '$w_{2}$',
 'acdb7b': '$w_{3}$',
 '6d360f': '$w_{4}$',
 'c264b9': '$w_{5}$',
 '272ddf': '$w_{6}$'}

In [22]:
out_plot = out.copy()
# out_plot["weight_label"] = out_plot["weight_hash"].map(hash_map).fillna(out_plot["weight_hash"])
 

fig = px.line(out_plot[['tau', 'count_norm', 'weight_label', 'alpha']], x='tau', y='count_norm', color='weight_label', facet_row='alpha', height=600, markers=False)



# Configurar eixo x para começar em 0.99
fig.update_xaxes(range=[1, None])
# fig.update_yaxes(range=[0.5, None])

# Deixar fundo branco e adicionar linhas dos eixos
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='black', zerolinewidth=1),
    yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='black', zerolinewidth=1)
)

# Aplicar para todos os subplots
for i in range(1, 6):
    fig.update_layout({
        f'xaxis{i}': dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='black', zerolinewidth=1),
        f'yaxis{i}': dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='black', zerolinewidth=1)
    })

fig.show()

In [23]:
taus = np.array([1.0, 1.5])  # ajuste aqui (pode adicionar mais)
ratio_cols = [f'ratio_{i}' for i in range(1, 6)]

df_long = df_min_clean.melt(
    id_vars=['weight_label', 'alpha', 'clientes', 'classe'],
    value_vars=ratio_cols,
    var_name='ratio',
    value_name='ratio_value'
).dropna(subset=['ratio_value'])

counts = (
    df_long.groupby(['alpha', 'clientes', 'classe', 'weight_label'])['ratio_value']
    .apply(lambda s: pd.Series({tau: (s <= tau).sum() for tau in taus}))
    .rename('count')
    .reset_index()
    .rename(columns={'level_4': 'tau'})  # se der mismatch, veja nota abaixo
)

# Escolhe o weight_label com maior count para cada (alpha, clientes, classe, tau)
# Desempate (opcional): se empatar no count, escolhe o weight_label "maior" (ou troque a regra)
winners = (
    counts.sort_values(['alpha', 'clientes', 'classe', 'tau', 'count', 'weight_label'])
    .groupby(['alpha', 'clientes', 'classe', 'tau'], as_index=False)
    .tail(1)
)

# Tabela final: linhas tau (e alpha), colunas repetidas por Cliente->Classe
table = winners.pivot_table(
    index=['alpha', 'tau'],
    columns=['clientes', 'classe'],
    values='weight_label',
    aggfunc='first'
).sort_index()

table

clientes      5.0                                 10.0                    \
classe         1.0      2.0      3.0      4.0      1.0      2.0      3.0   
alpha tau                                                                  
0.01  1.0  $w_{2}$  $w_{6}$  $w_{3}$  $w_{3}$  $w_{6}$  $w_{5}$  $w_{1}$   
      1.5  $w_{5}$  $w_{6}$  $w_{4}$  $w_{6}$  $w_{6}$  $w_{6}$  $w_{1}$   
0.99  1.0  $w_{1}$  $w_{5}$  $w_{2}$  $w_{3}$  $w_{2}$  $w_{2}$  $w_{2}$   
      1.5  $w_{1}$  $w_{5}$  $w_{1}$  $w_{3}$  $w_{2}$  $w_{2}$  $w_{2}$   

clientes            
classe         4.0  
alpha tau           
0.01  1.0  $w_{6}$  
      1.5  $w_{6}$  
0.99  1.0  $w_{6}$  
      1.5  $w_{6}$

In [24]:

def winners_to_latex_alpha_in_columns(
    winners: pd.DataFrame,
    taus_order=None,
    alpha_order=None,
    clientes_order=None,
    classes_order=None,
    class_roman_map=None,
    caption=None,
    label=None,
    table_env=True,
    debug=False,
):
    w = winners.copy()
    w['alpha'] = pd.to_numeric(w['alpha'], errors='coerce')
    w['tau'] = pd.to_numeric(w['tau'], errors='coerce')
    w['clientes'] = pd.to_numeric(w['clientes'], errors='ignore')
    w['classe'] = pd.to_numeric(w['classe'], errors='ignore')

    if alpha_order is None:
        alpha_order = sorted(pd.unique(w['alpha']).tolist())
    if clientes_order is None:
        clientes_order = sorted(pd.unique(w['clientes']).tolist())
    if classes_order is None:
        classes_order = sorted(pd.unique(w['classe']).tolist())

    # mapeamento default 1..4 -> I..IV
    if class_roman_map is None:
        class_roman_map = {1: "I", 2: "II", 3: "III", 4: "IV"}

    # matriz tau x (alpha, clientes, classe)
    mat = w.pivot_table(
        index='tau',
        columns=['alpha', 'clientes', 'classe'],
        values='weight_label',
        aggfunc='first'
    )

    # força ordem das colunas
    full_cols = pd.MultiIndex.from_product(
        [alpha_order, clientes_order, classes_order],
        names=['alpha', 'clientes', 'classe']
    )
    mat = mat.reindex(columns=full_cols)

    # força ordem de taus (linhas)
    if taus_order is None:
        taus_order = sorted(pd.unique(w['tau']).tolist())
    mat = mat.reindex(index=taus_order)

    if debug:
        print("mat shape:", mat.shape)
        print("non-null cells:", int(mat.notna().sum().sum()))
        print("alpha_order:", alpha_order)
        print("clientes_order:", clientes_order)
        print("classes_order:", classes_order)
        print("taus_order:", taus_order)

    if mat.dropna(how='all').empty:
        raise ValueError("Sem dados para os taus/alphas/clientes/classes escolhidos (mat ficou vazia).")

    A = len(alpha_order)
    C = len(clientes_order)
    K = len(classes_order)

    colspec = "l" + "c" * (A * C * K)

    def alpha_label(a):
        # evita notação científica tipo 1e-2
        return rf"$\alpha={a:g}$"

    def tau_label(t):
        return rf"$\rho(\tau)={t:g}$"

    def class_label(cl):
        # se cl não estiver no mapa, cai no próprio valor
        roman = class_roman_map.get(cl, str(cl))
        return roman

    lines = []
    if table_env:
        lines += [r"\begin{table}[t]", r"\centering"]
    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")

    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")

    # Header 1: blocos por alpha
    lines.append(
        r"\multicolumn{1}{c}{\textbf{}} & "
        + " & ".join([rf"\multicolumn{{{C*K}}}{{c}}{{\textbf{{{alpha_label(a)}}}}}" for a in alpha_order])
        + r" \\"
    )

    # cmidrule por alpha (começa na coluna 2)
    cmid = []
    start = 2
    for _ in alpha_order:
        end = start + (C * K) - 1
        cmid.append(rf"\cmidrule(lr){{{start}-{end}}}")
        start = end + 1
    lines.append(" ".join(cmid))

    # Header 2: clientes dentro de cada alpha
    row2 = [r"\multicolumn{1}{c}{\textbf{}}"]
    for _a in alpha_order:
        for c in clientes_order:
            row2.append(rf"\multicolumn{{{K}}}{{c}}{{\textbf{{clientes {int(c)}}}}}")
    lines.append(" & ".join(row2) + r" \\")

    # cmidrule por cliente
    cmid2 = []
    start = 2
    for _a in alpha_order:
        for _c in clientes_order:
            end = start + K - 1
            cmid2.append(rf"\cmidrule(lr){{{start}-{end}}}")
            start = end + 1
    lines.append(" ".join(cmid2))

    # Header 3: classes (I, II, III, IV)
    row3 = [r"\multicolumn{1}{c}{\textbf{}}"]
    for _a in alpha_order:
        for _c in clientes_order:
            for cl in classes_order:
                row3.append(rf"\multicolumn{{1}}{{c}}{{\textbf{{{class_label(cl)}}}}}")
    lines.append(" & ".join(row3) + r" \\")
    lines.append(r"\midrule")

    # Corpo: tau como rho(tau)=...
    for tau in mat.index.tolist():
        row = [tau_label(tau)]
        for a in alpha_order:
            for c in clientes_order:
                for cl in classes_order:
                    v = mat.loc[tau, (a, c, cl)]
                    row.append("" if pd.isna(v) else str(v))
        lines.append(" & ".join(row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if table_env:
        lines.append(r"\end{table}")

    return "\n".join(lines)

In [36]:
taus = [1.0, 1.5]
alpha_order = [0.01, 0.99]
clientes_order = [5, 10]   # ou [5] no teu exemplo
classes_order = [1, 2, 3, 4]

# você define a ordem:
# hash_list = ['6c465d', 'fc6e37', '4a1ce6', '82eac2', '7a4d83', '8f2f47']

# def make_weight_hash_map_from_list(hash_list, start_at=1):
    # return {h: rf"$w_{{{i}}}$" for i, h in enumerate(hash_list, start=start_at)}

# hash_map = make_weight_hash_map_from_list(hash_list)

winners_mapped = winners.copy()
# winners_mapped['weight_hash'] = winners_mapped['weight_hash'].map(hash_map).fillna(winners_mapped['weight_hash'])

latex_code = winners_to_latex_alpha_in_columns(
    winners_mapped,
    taus_order=taus,
    alpha_order=alpha_order,
    clientes_order=clientes_order,
    classes_order=classes_order,
    table_env=True,
    debug=True,
)
print(latex_code)

mat shape: (2, 16)
non-null cells: 32
alpha_order: [0.01, 0.99]
clientes_order: [5, 10]
classes_order: [1, 2, 3, 4]
taus_order: [1.0, 1.5]
\begin{table}[t]
\centering
\begin{tabular}{lcccccccccccccccc}
\toprule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{8}{c}{\textbf{$\alpha=0.01$}} & \multicolumn{8}{c}{\textbf{$\alpha=0.99$}} \\
\cmidrule(lr){2-9} \cmidrule(lr){10-17}
\multicolumn{1}{c}{\textbf{}} & \multicolumn{4}{c}{\textbf{clientes 5}} & \multicolumn{4}{c}{\textbf{clientes 10}} & \multicolumn{4}{c}{\textbf{clientes 5}} & \multicolumn{4}{c}{\textbf{clientes 10}} \\
\cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{I}} & \multicolumn{1}{c}{\textbf{II}} & \multicolumn{1}{c}{\textbf{III}} & \multicolumn{1}{c}{\textbf{IV}} & \multicolumn{1}{c}{\textbf{I}} & \multicolumn{1}{c}{\textbf{II}} & \multicolumn{1}{c}{\textbf{III}} & \multicolumn{1}{c}{\textbf{IV}} & \multicolumn{1}{c}{\textbf{I}} & \multi

/var/folders/8x/jrzgqn_143nb8h3yky8hy_d00000gn/T/ipykernel_22501/2091940973.py:16: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  w['clientes'] = pd.to_numeric(w['clientes'], errors='ignore')
/var/folders/8x/jrzgqn_143nb8h3yky8hy_d00000gn/T/ipykernel_22501/2091940973.py:17: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  w['classe'] = pd.to_numeric(w['classe'], errors='ignore')


In [37]:
def winners_alpha_tau_only(counts: pd.DataFrame, tie_breaker='weight_label'):
    # agrega (soma) count sobre clientes/classe
    agg = (
        counts.groupby(['alpha', 'tau', 'weight_label'], as_index=False)['count']
        .sum()
    )

    # escolhe weight_hash com maior count para cada (alpha, tau)
    sort_cols = ['alpha', 'tau', 'count']
    asc = [True, True, True]

    if tie_breaker is not None:
        sort_cols.append(tie_breaker)
        asc.append(True)  # desempate determinístico

    winners2 = (
        agg.sort_values(sort_cols, ascending=asc)
        .groupby(['alpha', 'tau'], as_index=False)
        .tail(1)
        .copy()
    )

    return winners2

In [38]:
def winners2_to_latex_alpha_cols_tau_rows(
    winners2: pd.DataFrame,
    taus_order=None,
    alpha_order=None,
    caption=None,
    label=None,
    table_env=True,
):
    w = winners2.copy()
    w['alpha'] = pd.to_numeric(w['alpha'], errors='coerce')
    w['tau'] = pd.to_numeric(w['tau'], errors='coerce')

    if alpha_order is None:
        alpha_order = sorted(pd.unique(w['alpha']).tolist())
    if taus_order is None:
        taus_order = sorted(pd.unique(w['tau']).tolist())

    mat = w.pivot_table(
        index='tau',
        columns='alpha',
        values='weight_label',
        aggfunc='first'
    ).reindex(index=taus_order, columns=alpha_order)

    if mat.dropna(how='all').empty:
        raise ValueError("Sem dados para os taus/alphas escolhidos.")

    colspec = "l" + "c" * len(alpha_order)

    def alpha_label(a):  # cabeçalho como você pediu
        return rf"$\alpha={a:g}$"

    def tau_label(t):    # linha como você pediu
        return rf"$\rho(\tau)={t:g}$"

    lines = []
    if table_env:
        lines += [r"\begin{table}[t]", r"\centering"]
    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")

    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")

    # header
    lines.append(
        r"\multicolumn{1}{c}{\textbf{}} & "
        + " & ".join([rf"\multicolumn{{1}}{{c}}{{\textbf{{{alpha_label(a)}}}}}" for a in alpha_order])
        + r" \\"
    )
    lines.append(r"\midrule")

    # body
    for tau in mat.index.tolist():
        row = [tau_label(tau)]
        for a in alpha_order:
            v = mat.loc[tau, a]
            row.append("" if pd.isna(v) else str(v))
        lines.append(" & ".join(row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    if table_env:
        lines.append(r"\end{table}")

    return "\n".join(lines)

In [47]:
winners2 = winners_alpha_tau_only(counts)

# hash_list = ['6c465d', 'fc6e37', '4a1ce6', '82eac2', '7a4d83', '8f2f47']
# hash_map = make_weight_hash_map_from_list(hash_list)
winners_mapped = winners2.copy()
# winners_mapped['weight_hash'] = winners_mapped['weight_hash'].map(hash_map).fillna(winners_mapped['weight_hash'])


latex_code = winners2_to_latex_alpha_cols_tau_rows(
    winners_mapped,
    taus_order=[1.0, 1.5],
    alpha_order=[0.01, 0.99],
    table_env=True
)
print(latex_code)

\begin{table}[t]
\centering
\begin{tabular}{lcc}
\toprule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{$\alpha=0.01$}} & \multicolumn{1}{c}{\textbf{$\alpha=0.99$}} \\
\midrule
$\rho(\tau)=1$ & $w_{6}$ & $w_{2}$ \\
$\rho(\tau)=1.5$ & $w_{6}$ & $w_{2}$ \\
\bottomrule
\end{tabular}
\end{table}


In [48]:
def count_wins_by_clientes_classe(
    winners: pd.DataFrame,
    clientes_order=None,
    classes_order=None,
):
    """
    Conta quantas vezes cada weight_label venceu para cada combinação de clientes e classe,
    agregando sobre todos os valores de tau.
    """
    w = winners.copy()
    w['clientes'] = pd.to_numeric(w['clientes'], errors='ignore')
    w['classe'] = pd.to_numeric(w['classe'], errors='ignore')
    
    if clientes_order is None:
        clientes_order = sorted(pd.unique(w['clientes']).tolist())
    if classes_order is None:
        classes_order = sorted(pd.unique(w['classe']).tolist())
    
    # Agregar contagens por (clientes, classe, weight_label) - soma sobre tau
    win_counts = (
        w.groupby(['clientes', 'classe', 'weight_label'], as_index=False)
        .size()
        .rename(columns={'size': 'win_count'})
    )
    
    return win_counts

In [49]:
def wins_table_to_latex(
    win_counts: pd.DataFrame,
    clientes_order=None,
    classes_order=None,
    weight_order=None,
    class_roman_map=None,
    caption=None,
    label=None,
    table_env=True,
):
    """
    Gera tabela LaTeX com linhas = (clientes, classe) e colunas = weight_label.
    Valores são a contagem de vitórias.
    """
    wc = win_counts.copy()
    wc['clientes'] = pd.to_numeric(wc['clientes'], errors='ignore')
    wc['classe'] = pd.to_numeric(wc['classe'], errors='ignore')
    
    if clientes_order is None:
        clientes_order = sorted(pd.unique(wc['clientes']).tolist())
    if classes_order is None:
        classes_order = sorted(pd.unique(wc['classe']).tolist())
    if weight_order is None:
        weight_order = sorted(pd.unique(wc['weight_label']).tolist())
    
    # mapeamento default 1..4 -> I..IV
    if class_roman_map is None:
        class_roman_map = {1: "I", 2: "II", 3: "III", 4: "IV"}
    
    # Pivot: linhas = (clientes, classe), colunas = weight_label
    mat = wc.pivot_table(
        index=['clientes', 'classe'],
        columns='weight_label',
        values='win_count',
        aggfunc='sum',
        fill_value=0
    )
    
    # Reindexar para garantir ordem
    full_rows = pd.MultiIndex.from_product(
        [clientes_order, classes_order],
        names=['clientes', 'classe']
    )
    mat = mat.reindex(index=full_rows, columns=weight_order, fill_value=0)
    
    # Adicionar linha "all classes" para cada clientes
    all_classes_rows = []
    for c in clientes_order:
        row_sum = mat.loc[c].sum(axis=0)
        all_classes_rows.append(((c, 'all'), row_sum))
    
    # Criar DataFrame para "all classes"
    all_classes_df = pd.DataFrame(
        [row[1] for row in all_classes_rows],
        index=pd.MultiIndex.from_tuples([row[0] for row in all_classes_rows], names=['clientes', 'classe'])
    )
    
    # Concatenar com a matriz original
    mat_with_all = pd.concat([mat, all_classes_df]).sort_index()
    
    # Adicionar linha de proporção (total geral)
    total_wins = mat_with_all.sum(axis=0)
    grand_total = total_wins.sum()
    proportions = total_wins / grand_total if grand_total > 0 else total_wins * 0
    
    # Construir LaTeX
    num_weights = len(weight_order)
    colspec = "l" + "c" * num_weights
    
    def class_label(cl):
        if cl == 'all':
            return "all classes"
        roman = class_roman_map.get(cl, str(cl))
        return roman
    
    lines = []
    if table_env:
        lines += [r"\begin{table}[t]", r"\centering"]
    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")
    
    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    
    # Header: pesos
    header = [r"\multicolumn{1}{c}{\textbf{}}"] + [rf"\multicolumn{{1}}{{c}}{{\textbf{{{w}}}}}" for w in weight_order]
    lines.append(" & ".join(header) + r" \\")
    lines.append(r"\midrule")
    
    # Corpo: linhas por clientes e classe
    prev_clientes = None
    for (c, cl) in mat_with_all.index:
        # Adicionar separador entre diferentes números de clientes
        if prev_clientes is not None and c != prev_clientes:
            lines.append(r"\midrule")
        prev_clientes = c
        
        # Label da linha
        if cl == 'all':
            row_label = class_label(cl)
        else:
            row_label = class_label(cl)
        
        # Valores
        row = [row_label]
        for w in weight_order:
            val = mat_with_all.loc[(c, cl), w]
            row.append(str(int(val)) if val != 0 else "0")
        lines.append(" & ".join(row) + r" \\")
    
    lines.append(r"\midrule")
    
    # Linha de proporção
    prop_row = ["proportion"]
    for w in weight_order:
        prop_row.append(f"{proportions[w]:.2f}")
    lines.append(" & ".join(prop_row) + r" \\")
    
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if table_env:
        lines.append(r"\end{table}")
    
    return "\n".join(lines)

In [50]:
def wins_table_to_latex(
    win_counts: pd.DataFrame,
    clientes_order=None,
    classes_order=None,
    weight_order=None,
    class_roman_map=None,
    caption=None,
    label=None,
    table_env=True,
):
    """
    Gera tabela LaTeX com linhas = (clientes, classe) e colunas = weight_label.
    Valores são a contagem de vitórias.
    """
    wc = win_counts.copy()
    wc['clientes'] = pd.to_numeric(wc['clientes'], errors='ignore')
    wc['classe'] = pd.to_numeric(wc['classe'], errors='ignore')
    
    if clientes_order is None:
        clientes_order = sorted(pd.unique(wc['clientes']).tolist())
    if classes_order is None:
        classes_order = sorted(pd.unique(wc['classe']).tolist())
    if weight_order is None:
        weight_order = sorted(pd.unique(wc['weight_label']).tolist())
    
    # mapeamento default 1..4 -> I..IV
    if class_roman_map is None:
        class_roman_map = {1: "I", 2: "II", 3: "III", 4: "IV"}
    
    # Pivot: linhas = (clientes, classe), colunas = weight_label
    mat = wc.pivot_table(
        index=['clientes', 'classe'],
        columns='weight_label',
        values='win_count',
        aggfunc='sum',
        fill_value=0
    )
    
    # Reindexar para garantir ordem
    full_rows = pd.MultiIndex.from_product(
        [clientes_order, classes_order],
        names=['clientes', 'classe']
    )
    mat = mat.reindex(index=full_rows, columns=weight_order, fill_value=0)
    
    # Adicionar linha "All instances" para cada clientes
    all_classes_rows = []
    for c in clientes_order:
        row_sum = mat.loc[c].sum(axis=0)
        all_classes_rows.append(((c, 'all'), row_sum))
    
    # Criar DataFrame para "All instances"
    all_classes_df = pd.DataFrame(
        [row[1] for row in all_classes_rows],
        index=pd.MultiIndex.from_tuples([row[0] for row in all_classes_rows], names=['clientes', 'classe'])
    )
    
    # Concatenar com a matriz original
    mat_with_all = pd.concat([mat, all_classes_df]).sort_index()
    
    # Adicionar linha de proporção (total geral)
    total_wins = mat_with_all.sum(axis=0)
    grand_total = total_wins.sum()
    proportions = total_wins / grand_total if grand_total > 0 else total_wins * 0
    
    # Construir LaTeX
    num_weights = len(weight_order)
    # Adiciona coluna extra para "Clientes"
    colspec = "l" + "l" + "c" * num_weights
    
    def class_label(cl):
        if cl == 'all':
            return "All instances"
        roman = class_roman_map.get(cl, str(cl))
        return roman
    
    lines = []
    if table_env:
        lines += [r"\begin{table}[t]", r"\centering"]
    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")
    
    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    
    # Linha superior: "# the best performing weight" - span todas as colunas
    lines.append(rf"\multicolumn{{{num_weights + 2}}}{{c}}{{\# the best performing weight}} \\")
    lines.append(r"\midrule")
    
    # Header: coluna vazia para "Clientes", coluna vazia para classe, depois pesos
    header = [r"\multicolumn{1}{c}{\textbf{}}", r"\multicolumn{1}{c}{\textbf{}}"] + [rf"\multicolumn{{1}}{{c}}{{\textbf{{{w}}}}}" for w in weight_order]
    lines.append(" & ".join(header) + r" \\")
    lines.append(r"\midrule")
    
    # Corpo: linhas por clientes e classe
    for c in clientes_order:
        # Pegar todas as linhas para este cliente (classes + all instances)
        client_rows = [(c, cl) for cl in classes_order] + [(c, 'all')]
        num_rows = len(client_rows)
        
        for idx, (_, cl) in enumerate(client_rows):
            # Primeira linha do grupo: adiciona multirow para clientes
            if idx == 0:
                row = [rf"\multirow{{{num_rows}}}{{*}}{{\textbf{{{int(c)} clients}}}}"]
            else:
                row = [""]  # Células vazias para as outras linhas do multirow
            
            # Label da classe
            row.append(class_label(cl))
            
            # Valores
            for w in weight_order:
                val = mat_with_all.loc[(c, cl), w]
                row.append(str(int(val)) if val != 0 else "0")
            
            lines.append(" & ".join(row) + r" \\")
        
        # Adicionar midrule após cada grupo de clientes (exceto o último)
        if c != clientes_order[-1]:
            lines.append(r"\midrule")
    
    lines.append(r"\midrule")
    
    # Linha de proporção - span nas duas primeiras colunas
    prop_row = [rf"\multicolumn{{2}}{{l}}{{\textbf{{Proportion}}}}"]
    for w in weight_order:
        prop_row.append(f"{proportions[w]:.2f}")
    lines.append(" & ".join(prop_row) + r" \\")
    
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if table_env:
        lines.append(r"\end{table}")
    
    return "\n".join(lines)

In [51]:
# Gerar contagem de vitórias por clientes e classe
win_counts = count_wins_by_clientes_classe(winners)

# Ordenar pesos conforme hash_map
weight_order = [f"$w_{{{i}}}$" for i in range(1, 7)]  # w_1 até w_6
clientes_order = [5, 10]
classes_order = [1, 2, 3, 4]

# Gerar código LaTeX
latex_code = wins_table_to_latex(
    win_counts,
    clientes_order=clientes_order,
    classes_order=classes_order,
    weight_order=weight_order,
    caption="Global performance of a portion of weight space $W$",
    label="tab:weight_performance",
    table_env=True,
)

print(latex_code)

\begin{table}[t]
\centering
\caption{Global performance of a portion of weight space $W$}
\label{tab:weight_performance}
\begin{tabular}{llcccccc}
\toprule
\multicolumn{8}{c}{\# the best performing weight} \\
\midrule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{$w_{1}$}} & \multicolumn{1}{c}{\textbf{$w_{2}$}} & \multicolumn{1}{c}{\textbf{$w_{3}$}} & \multicolumn{1}{c}{\textbf{$w_{4}$}} & \multicolumn{1}{c}{\textbf{$w_{5}$}} & \multicolumn{1}{c}{\textbf{$w_{6}$}} \\
\midrule
\multirow{5}{*}{\textbf{5 clients}} & I & 2 & 1 & 0 & 0 & 1 & 0 \\
 & II & 0 & 0 & 0 & 0 & 2 & 2 \\
 & III & 1 & 1 & 1 & 1 & 0 & 0 \\
 & IV & 0 & 0 & 3 & 0 & 0 & 1 \\
 & All instances & 3 & 2 & 4 & 1 & 3 & 3 \\
\midrule
\multirow{5}{*}{\textbf{10 clients}} & I & 0 & 2 & 0 & 0 & 0 & 2 \\
 & II & 0 & 2 & 0 & 0 & 1 & 1 \\
 & III & 2 & 2 & 0 & 0 & 0 & 0 \\
 & IV & 0 & 0 & 0 & 0 & 0 & 4 \\
 & All instances & 2 & 6 & 0 & 0 & 1 & 7 \\
\midrule
\multicolumn{2}{l}{\textbf{Propor

/var/folders/8x/jrzgqn_143nb8h3yky8hy_d00000gn/T/ipykernel_22501/2898035697.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  w['clientes'] = pd.to_numeric(w['clientes'], errors='ignore')
/var/folders/8x/jrzgqn_143nb8h3yky8hy_d00000gn/T/ipykernel_22501/2898035697.py:12: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  w['classe'] = pd.to_numeric(w['classe'], errors='ignore')
/var/folders/8x/jrzgqn_143nb8h3yky8hy_d00000gn/T/ipykernel_22501/100339556.py:16: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  wc['clientes'] = pd.to_numeric(wc['clientes'], errors='ignore')
/var/folders/8x/jrzgqn_143nb8h3yky8hy_d00000gn/T/ipykernel_22501/100339556.py:1

In [52]:
# Criar mapeamento usando os hashes que realmente existem no df atual
current_hash_list = df["weight_hash"].unique()
current_hash_map = make_weight_hash_map_from_list(current_hash_list, start_at=1)

print("Hash map atual (do df):")
print(current_hash_map)

# Pegar os vetores de peso únicos do dataframe
weight_mapping = df[['weight_hash', 'weight']].drop_duplicates()

# Criar dicionário: weight_label -> vetor de pesos
label_to_vector = {}
for _, row in weight_mapping.iterrows():
    hash_val = row['weight_hash']
    if hash_val in current_hash_map:
        label = current_hash_map[hash_val]
        # Converter string para lista
        vector = ast.literal_eval(row['weight'])
        label_to_vector[label] = vector

# Ordenar por label
sorted_labels = sorted(label_to_vector.keys(), key=lambda x: int(x.split('_')[1].strip('{}$')))

print("\nMapeamento de pesos:")
for label in sorted_labels:
    print(f"{label}: {label_to_vector[label]}")

Hash map atual (do df):
{'d086fe': '$w_{1}$', 'edcc2b': '$w_{2}$', 'acdb7b': '$w_{3}$', '6d360f': '$w_{4}$', 'c264b9': '$w_{5}$', '272ddf': '$w_{6}$'}

Mapeamento de pesos:
$w_{1}$: [0.3500000000000001, 0.3500000000000001, 0.1, 0.1, 0.1]
$w_{2}$: [0.2, 0.5, 0.1, 0.1, 0.1]
$w_{3}$: [0.2, 0.2, 0.2, 0.2, 0.2]
$w_{4}$: [0.25, 0.25, 0.25, 0.125, 0.125]
$w_{5}$: [0.29999999999999993, 0.29999999999999993, 0.1000000000000002, 0.19999999999999996, 0.1]
$w_{6}$: [0.29999999999999993, 0.29999999999999993, 0.1000000000000002, 0.1, 0.19999999999999996]


In [53]:
def weight_mapping_to_latex(
    label_to_vector: dict,
    caption=None,
    label=None,
    table_env=True,
    decimal_places=3,
):
    """
    Gera tabela LaTeX mostrando o mapeamento entre weight labels e seus vetores.
    Linhas: weight labels (w_1, w_2, ...)
    Colunas: componentes do vetor (v_1, v_2, v_3, v_4, v_5)
    """
    # Ordenar labels
    sorted_labels = sorted(label_to_vector.keys(), key=lambda x: int(x.split('_')[1].strip('{}$')))
    
    # Número de componentes (assumindo que todos os vetores têm o mesmo tamanho)
    num_components = len(list(label_to_vector.values())[0])
    
    # Construir LaTeX
    colspec = "l" + "c" * num_components
    
    lines = []
    if table_env:
        lines += [r"\begin{table}[t]", r"\centering"]
    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")
    
    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    
    # Header: componentes do vetor
    header = [r"\multicolumn{1}{c}{\textbf{}}"] + [rf"\multicolumn{{1}}{{c}}{{\textbf{{$v_{{{i}}}$}}}}" for i in range(1, num_components + 1)]
    lines.append(" & ".join(header) + r" \\")
    lines.append(r"\midrule")
    
    # Corpo: cada linha é um weight label com seus valores arredondados
    for wlabel in sorted_labels:
        vector = label_to_vector[wlabel]
        row = [wlabel] + [f"{v:.{decimal_places}f}" for v in vector]
        lines.append(" & ".join(row) + r" \\")
    
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if table_env:
        lines.append(r"\end{table}")
    
    return "\n".join(lines)

In [54]:
# Gerar código LaTeX para tabela de mapeamento de pesos
latex_weight_mapping = weight_mapping_to_latex(
    label_to_vector,
    caption="Weight vector mapping",
    label="tab:weight_mapping",
    table_env=True,
    decimal_places=3,
)

print(latex_weight_mapping)

\begin{table}[t]
\centering
\caption{Weight vector mapping}
\label{tab:weight_mapping}
\begin{tabular}{lccccc}
\toprule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{$v_{1}$}} & \multicolumn{1}{c}{\textbf{$v_{2}$}} & \multicolumn{1}{c}{\textbf{$v_{3}$}} & \multicolumn{1}{c}{\textbf{$v_{4}$}} & \multicolumn{1}{c}{\textbf{$v_{5}$}} \\
\midrule
$w_{1}$ & 0.350 & 0.350 & 0.100 & 0.100 & 0.100 \\
$w_{2}$ & 0.200 & 0.500 & 0.100 & 0.100 & 0.100 \\
$w_{3}$ & 0.200 & 0.200 & 0.200 & 0.200 & 0.200 \\
$w_{4}$ & 0.250 & 0.250 & 0.250 & 0.125 & 0.125 \\
$w_{5}$ & 0.300 & 0.300 & 0.100 & 0.200 & 0.100 \\
$w_{6}$ & 0.300 & 0.300 & 0.100 & 0.100 & 0.200 \\
\bottomrule
\end{tabular}
\end{table}
